<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Sequential_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch

In [ ]:
torch.manual_seed(42)

In [ ]:
class Linear():
  def __init__(self, input_dim, output_dim, set_bias=False):
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.weights = torch.randn((input_dim, output_dim))
    self.bias = torch.ones((output_dim)) if set_bias else None

  def __call__(self, x):
    self.out = x @ self.weights + self.bias if self.bias is not None else x @ self.weights
    return self.out

  def __repr__(self):
    return f"Linear(input_dim={self.input_dim}, output_dim={self.output_dim})"

  def parameters(self):
    return [self.weights] + ([] if self.bias is None else [self.bias])


class Tanh():
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out

  def __repr__(self):
    return f"Tanh()"

  def parameters(self):
    return []

class Sequential():
  def __init__(self, layers):
    self.layers = layers
    self.params = []

  def __repr__(self):
    s = []
    for layer in self.layers:
      s.append(layer)
    return f"Sequential(\n{s}\n)"

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out

  def parameters(self):
    self.params = [p for layer in self.layers for p in layer.parameters()]
    return self.params

In [ ]:
class BasicRNN():
  def __init__(self, n_features, n_neurons):
    self.hidden = n_neurons
    self.input = n_features
    self.W_x = torch.randn((n_features, n_neurons))
    self.W_h = torch.randn((n_neurons, n_neurons))
    self.b_h = torch.zeros((n_neurons))

  def __call__(self, x, h_x=None):
    self.state = h_x or torch.zeros((x.shape[-2], self.hidden))

    output = []
    #would have to check this out
    for input in x:
      self.state = torch.tanh(
          input @ self.W_x +
          self.state @ self.W_h +
          self.b_h
      )
      output.append(self.state)

    return output, self.state


  def __repr__(self):
    return f"BasicRNN(n_inputs={self.input}, n_output={self.hidden})"


  def parameters(self):
    return [self.W_x, self.W_h, self.b_h]








In [ ]:
class BiDirectionalRNN():
  def __init__(self, n_hidden, n_in):
    self.f_rnn = BasicRNN(n_hidden)
    self.b_rnn = BasicRNN(n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, x, state=None):
    x_reverse = torch.flip(x, dims=[0])
    f_h, b_h = state if state else (None, None)

    output_f, state_f = self.f_rnn(x, f_h)
    output_b, state_b = self.b_rnn(x_reverse, b_h)

    output_b = torch.flip(output_b, dims=[0])
    outputs = torch.cat((output_f, output_b), dim=-1)

    return outputs, (f_h, b_h)


  def parameters(self):
    return self.f_rnn.parameters() + self.b_rnn.parameters()

  def __repr__(self):
    return f"BiDirectionalRNN(n_hidden={self.n_hidden})"


In [ ]:
class LSTMCell():
  def __init__(self, n_hidden):
    self.n_hidden = n_hidden

  def __call__(self, input, state=None):
    self.c, self.h_x = state or (torch.ones((self.in_x.shape[-2], self.n_hidden)),
                                 torch.zeros((x.shape[-2], self.n_hidden)))
    long_term_outputs = []
    short_term_outputs = []

    for x in input:
      self.in_x = torch.concat((self.h_x, x), dim=-1)

      #forget gate
      self.W_f = torch.randn((self.in_x.shape[-1], self.n_hidden))
      self.b_f = torch.zeros(self.n_hidden)
      self.f_t = torch.sigmoid(self.in_x @ self.W_f + self.b_f)
      self.c *= self.f_t

      #input gate
      self.W_i = torch.randn((self.in_x.shape[-1], self.n_hidden))
      self.b_i = torch.zeros(self.n_hidden)
      self.i_t = torch.sigmoid(self.in_x @ self.W_f + self.b_i)
      self.W_c = torch.randn((self.in_x.shape[-1], self.n_hidden))
      self.b_c = torch.zeros(self.n_hidden)
      self.c_t = torch.tanh(self.in_x @ self.W_c + self.b_c)
      self.i_prod = self.i_t * self.c_t
      self.c += self.i_prod

      #output gate
      self.W_o = torch.randn((self.in_x.shape[-1], self.n_hidden))
      self.b_o = torch.zeros(self.n_hidden)
      self.o_t = torch.sigmoid(self.in_x @ self.W_o + self.b_o)
      self.ogo = torch.tanh(self.c) * self.o_t
      self.h_x = self.ogo
      long_term_outputs.append(self.c)
      short_term_outputs.append(self.h_x)

    return long_term_outputs, short_term_outputs, self.c, self.h_x

  def parameters(self):
    return [self.W_f, self.b_f, self.W_i, self.b_i, self.W_c, self.b_c, self.W_o, self.b_o]

  def __repr__(self):
    return f"LSTM()"


In [ ]:
class BiDirectionalLSTM():
  def __init__(self, n_hidden):
    self.f_lstm = LSTMCell(n_hidden)
    self.b_lstm = LSTMCell(n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, x, state=None):
    (f_c, f_h), (b_c, b_h) = state if state else (None, None), (None, None)
    cf, hf = self.f_lstm(x, (f_c, f_h))
    x_reverse = torch.flip(x, [0])
    cb, hb = self.r_lstm(x_reverse, [0])
    cb, hb = torch.flip(cb, [0]), torch.flip(hb, [0])
    #might have to concatenate; not sure yet
    return (cf, cb), (hf, hb)

  def __repr__(self):
    return f"BiDirectionalLSTM(n_hidden={self.n_hidden})"

  def parameters(self):
    return self.f_lstm.parameters() + self.b_lstm.parameters()





In [ ]:
class GRUCell():
  def __init__(self, n_hidden):
    self.hidden = n_hidden

  def __call__(self, x, state=None):
    self.h_x = state or torch.zeros((x.shape[-2], self.n_hidden))
    self.in_x = torch.concat((self.h_x, x), dim=-1)

    #reset gate
    self.W_r = torch.randn((self.in_x.shape[-1], self.n_hidden))
    self.r_t = torch.sigmoid(self.in_x @ self.W_r)

    #update gate
    self.W_z = torch.randn((self.in_x.shape[-1], self.n_hidden))
    self.z_t = torch.sigmoid(self.in_x @ self.W_z)

    #candidate hidden state
    self.W_h = torch.randn((self.in_x.shape[-1], self.n_hidden))
    self.b_h = torch.ones(self.n_hidden)
    self.had = self.r_t * self.h_x
    self.inter = torch.concat((self.had, x), dim=-1)
    self.h_t = torch.tanh(self.inter @ self.W_h + self.b_h)

    #hidden state
    self.h_x = (
        (torch.ones_like(self.z_t) - self.z_t)
        * self.h_x
        )
    + self.z_t * self.h_t

    return self.h_x

  def __repr__(self):
    return f"GRU(n_hidden={self.n_hidden})"

  def parameters(self):
    return [self.W_r, self.W_z, self.W_h, self.b_h]


In [ ]:
class BiDirectionalGRU():
  def __init__(self, n_hidden):
    self.f_gru = GRUCell(n_hidden)
    self.b_gru = GRUCell(n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, x, state=None):
    f_h, b_h = state if state else (None, None)
    output_f = self.f_gru(x, f_h)
    x_rev = torch.flip(x, [0])
    #would also have to check if the state should also be reversed before being
    #being fed to the model. As of now, it isn't
    output_b = self.b_gru(x_rev, b_h)
    output_b = torch.flip(output_b, [0])
    return (output_f, output_b)

  def __repr__(self):
    return f"BiDirectionalGGRU(n_hidden={self.n_hidden})"

  def parameters(self):
    return self.f_gru.parameters() + self.b_gru.parameters()


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dataset_path = "/content/drive/MyDrive/Electric_Production.csv"

Mounted at /content/drive


In [ ]:
import pandas as pd
dataset = pd.read_csv(dataset_path)
prod_data = dataset["IPG2211A2N"]
prod_data = prod_data.values
prod_data.shape

(397,)

In [ ]:
seq_len = 30
seqs = []
targets = []
idx = prod_data.shape[0] - seq_len

for i in range(idx):
  seqs.append(prod_data[i: i+seq_len])
  targets.append([prod_data[i+seq_len]])

seqs = torch.tensor(np.array(seqs), dtype=torch.float32)
targets = torch.tensor(np.array(targets), dtype=torch.float32)

In [ ]:
seqs.shape
targets.shape


torch.Size([367, 1])

In [ ]:
seqs = seqs.unsqueeze(-1)
seqs = seqs.permute(1, 0, 2)
seqs.shape

torch.Size([30, 367, 1])

In [ ]:
print(seqs)

print(targets)


In [ ]:
rnn = BasicRNN(1, 10)
linear = Linear(10, 1)

In [ ]:
import torch.nn.functional as F

In [ ]:
epochs = 100
params = rnn.parameters() + linear.parameters()
for p in params:
  p.requires_grad = True

for i in range(epochs):
  #forward pass
  _, output = rnn(seqs)
  logits = linear(output)
  loss = F.mse_loss(logits, targets)

  #backward pass
  for p in params:
    p.grad = None

  loss.backward(retain_graph=True)

  lr = 0.08
  for p in params:
    p.data += -lr*p.grad

print(loss.item())


nan


In [ ]:
lstm = LSTMCell(10)
lstm_linear = Linear(10, 1)

In [ ]:
epochs = 100
lstm_params = lstm.parameters() + lstm_linear.parameters()
for p in lstm_params:
  p.requires_grad = True

for i in range(epochs):
  #forward pass
  _,_,_, output = rnn(seqs)
  logits = linear(output)
  loss = F.mse_loss(logits, targets)

  #backward pass
  for p in params:
    p.grad = None

  loss.backward(retain_graph=True)

  lr = 0.08
  for p in params:
    p.data += -lr*p.grad

print(loss.item())
